# COMP53415 – Assembly of 2D Puzzles – Test
**File:** `username-assembly-test.ipynb` (required)

This notebook **must** contain:
- Your **final puzzle-solving pipeline** (loading pieces, predicting poses, writing outputs).
- Your **evaluation code** (metrics on the provided splits).
- Your **mini-report**, written in Markdown, in the two sections below:

1. **Solution Design (≈250 words)**
2. **Analysis (≈250 words)**

Replace `username` in the filename with your CIS username before submission.


## State the type of your solution
In the cell below put one word either "Optimisation" or "Learning" (without speech marks) 

Optimisation

## 1. Solution Design (250 words)

In this section, clearly describe the overall design of your puzzle-solving system.  
Focus on *what you built* and *why you made these design choices*.

Recommended points to cover:

- **Overall pipeline**: from input pieces → feature extraction / cues → pose prediction → final arrangement.
- **Core techniques used**: e.g., texture or colour similarity, edge/line detection, shape/contour analysis, small learned features, object/motif cues, global reasoning.
- **Search / optimisation strategy**: e.g., greedy matching, beam search, genetic algorithm, ICP-style refinement, neighbour pruning.
- **Generalisation**: how the same approach works for both MS-COCO tile puzzles and DAFNE fragments **without dataset-specific conditionals**.
- **Constraints**: how you stayed within the ≤ 15M trainable parameters and ≤ 120s per group inference budget.

Keep this section concise and clear (around 250 words).


- **Overall pipeline**  Read puzzle groups from ZIP files, convert fragments to RGB arrays, parse canvas size from parameter text, compute pairwise compatibility, place fragments, and write one pose file per puzzle key.
- **Core techniques and cues**  Use HOG for texture structure, LAB histogram for colour distribution, and four side edge descriptors for boundary consistency. Compatibility score combines weighted feature distance and side to side edge matching.
- **Search and optimisation strategy**  Use contour area as the first seed cue and feature norm as fallback. For small puzzle size up to `STAGE3_FAST_PATH_MAX_N`, run greedy placement with four rotation candidates and neighbour based scoring. For larger puzzle size, switch to fast grid fallback with combined feature ordering.
- **Generalisation across datasets**  Keep the same feature extraction and placement logic for COCO tiles and DAFNE fragments. No separate model, no dataset specific parameter tuning in the prediction logic.
- **Constraint handling and runtime design**  Use pure optimisation with zero trainable parameters, disable beam search for stability, run timing three times, and report device and average runtime. Include dataset name in output filenames to prevent cross dataset key conflicts.

## 2. Analysis (250 words)

Provide a critical reflection on the performance and behaviour of your system across:

- MS-COCO puzzles: simple, moderate, and hard variants.
- DAFNE archaeological fragment puzzles.

Recommended points to cover:

- **Metric trends**: discuss RMSE-T, RMSE-R across datasets and difficulty levels.
- **Where your method works well**: examples of successful reconstructions and why they succeed.
- **Where your method struggles**: failure cases (e.g., heavy noise, ambiguous backgrounds, eroded edges, rotation ambiguities) and why.
- **Cross-dataset generalisation**: how well the same method worked on COCO vs DAFNE.
- **Limitations & future work**: what you would improve with more time or resources.

Aim for 250 words, focusing on insight rather than repetition.


**Quantitative insights.** The current results show clear domain gaps. COCO keeps lower translation error than Dafne, while Dafne remains much harder because fragments are irregular and visual evidence is weaker. Rotation error is also much higher on Moderate and Hard than on Simple. This matches the design choice of four rotation candidates only, which is fast but coarse for difficult cases. Neighbour F1 is moderate, which means local neighbourhood structure is partly recovered but still unstable in ambiguous regions.

**Where the method succeeds.** The pipeline works better when pieces have strong texture contrast, stable colour patterns, and clear borders. In these cases, HOG, LAB histogram, and side edge matching provide consistent compatibility signals. Performance is also stronger on smaller puzzles where the stage three placement with rotation is still active.

**Where it struggles.** The method degrades when noise, erosion, and repeated visual patterns reduce discriminative cues. Large puzzles are pushed to the fast grid fallback once piece count exceeds STAGE3 FAST PATH MAX N set to 25. That choice improves speed but weakens global matching quality and can increase translation error. Rotation quality is also bounded by discrete angle candidates.

**Cross dataset generalisation.** One unchanged optimisation pipeline is applied to both COCO and Dafne. The code does not switch to a separate model or separate feature extractor per dataset.

**Runtime, constraints, and next steps.** The design uses zero trainable parameters and reports three runs with device information and group level timing. In my current interpretation, one group means one full pipeline execution over one dataset split. With three COCO difficulty splits and one Dafne split, the total runtime is about 172 seconds, so the average per group is around 40 seconds. Future work will tune multiple controls instead of only `STAGE3_FAST_PATH_MAX_N`. First, enable eight angle candidates on selected puzzles where rotation ambiguity is high. Second, re enable beam search for small and medium puzzle sizes to improve global consistency. Third, use adaptive switching based on puzzle size and early runtime estimates so expensive search is only used where it brings clear gains. Finally, add a lightweight local refinement step after initial placement to reduce translation error without large runtime overhead.

## 3. Time Script (Do not remove)
This cell records the start time of the entire script.

We use this timestamp to measure how long the notebook takes to run from start to finish. This is useful for: 
 - Comparing training speed across different model sizes
 - Understanding the computational cost of different design choices
 - Debugging performance issues

The value stored in script_start is used later to compute the total runtime.

⚠️ Important rules:
 - Do not remove this cell.
 - Do not change the variable name script_start.
 - Do not move this cell lower in the notebook.

If you modify or rerun this cell after training has started, the reported runtime will be incorrect.

The code below captures a high-resolution timestamp using time.perf_counter().

In [ ]:
import time
from typing import Final
script_start: Final = time.perf_counter() # Do not remove or change this value

## 4. Setup and Imports

Configure paths and import dependencies. Edit `DATA_ROOT` and other paths to match your environment.


In [ ]:
import sys
from pathlib import Path
import math
import cv2
import torch
import numpy as np
from collections import defaultdict

DEVICE = torch.device("cpu")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.mps.is_available():
    DEVICE = torch.device("mps")
print("device:      ", DEVICE)

PROJECT_DIR = Path.cwd().resolve()
PROJECT_DIR_STR = str(PROJECT_DIR)
if PROJECT_DIR_STR not in sys.path:
    sys.path.insert(0, PROJECT_DIR_STR)

DATA_ROOT = PROJECT_DIR / "dataset"
OUTPUT_DIR = PROJECT_DIR / "predictions"
COCOTILES_ZIP = DATA_ROOT / "CocoTiles.zip"
AUGMENTATION = "None"                 # Options: "None", "Simple", "Moderate", "Hard"
DAFNE_ZIP = DATA_ROOT / "Dafne.zip"


FRAGMENT_SIZE = (24, 24)
NUM_WORKERS = 0
USE_ONE_COCO_DIFFICULTY_FOR_TIMED_RUNS = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ZIP = COCOTILES_ZIP
# Configuration summary (sanity check)
print("DATA_ROOT:      ", DATA_ROOT)
print("COCOTILES_ZIP:  ", COCOTILES_ZIP)
print("DAFNE_ZIP:      ", DAFNE_ZIP)
print("AUGMENTATION:   ", AUGMENTATION)
print("================")
print("RUNNING ON      ", DATASET_ZIP)

## 5. Dataset loading

In [ ]:
from UnifiedPuzzleSetZipDataset import make_unified_puzzle_dataloader_zip


def build_loader(zip_file, split_name="val", *, augment_mode="None", do_shuffle=False):
    return make_unified_puzzle_dataloader_zip(
        zip_path=str(zip_file),
        split=split_name,
        batch_size=1,
        shuffle=do_shuffle,
        num_workers=NUM_WORKERS,
        fixed_size=FRAGMENT_SIZE,
        normalize_rgb=True,
        return_optional_images=False,
        augment_mode=augment_mode,
        normalize_xy01=True,
    )


def make_val_loaders():
    loaders = {}
    
    if DATASET_ZIP == COCOTILES_ZIP:
        loaders[f"CocoTiles_{AUGMENTATION}"] = build_loader(COCOTILES_ZIP, augment_mode=AUGMENTATION)
    else:
        loaders["Dafne"] = build_loader(DAFNE_ZIP)
        
    return loaders


val_loaders = make_val_loaders()

## 6. Load additional data

e.g. Model


In [ ]:
# Todo: Add your code here

## 7. Your Predictor Code

In [ ]:
CELL_SIZE = 24

ANGLES_4 = [0, 90, 180, 270]
ANGLES_8 = [0, 45, 90, 135, 180, 225, 270, 315]
USE_8_ANGLES = False

STAGE3_FAST_PATH_MAX_N = 25  # above this use grid-only to stay under 120s
# tried 36 a lit bit more than 120s

BEAM_WIDTH = 2
BEAM_SEARCH_MAX_N = 0         # 0 = disable beam (greedy only) to meet 120s

W_HOG, W_COLOR, W_EDGE = 1.0, 0.3, 0.5
EDGE_COMPAT_WEIGHT = 0.4

def layout_rows_cols(n, W, H):
    if n <= 0:
        return 1, 1

    if W >= CELL_SIZE and H >= CELL_SIZE:
        c_cols = max(1, round(W / CELL_SIZE))
        c_rows = max(1, round(H / CELL_SIZE))
        if c_cols * c_rows == n:
            return c_rows, c_cols

    cols = max(1, round(math.sqrt(n)))
    rows = max(1, (n + cols - 1) // cols)
    return rows, cols

def contour_area(rgb):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, bin_ = cv2.threshold(g, 1, 255, cv2.THRESH_BINARY)
    cnts, _ = cv2.findContours(bin_, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not cnts:
        return 0.0
    return max(cv2.contourArea(c) for c in cnts)

def hog_descriptor(rgb, win=(24, 24), cell=(4, 4), block=(2, 2)):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) if rgb.ndim == 3 else np.asarray(rgb, dtype=np.uint8)
    h, w = g.shape[0], g.shape[1]
    if h < 8 or w < 8:
        return None
    try:
        d = cv2.HOGDescriptor(
            _winSize=(min(win[0], w), min(win[1], h)),
            _blockSize=(block[0] * cell[0], block[1] * cell[1]),
            _blockStride=(cell[0], cell[1]), _cellSize=cell, _nbins=9
        ).compute(g)
        return d.flatten().astype(np.float32) if d is not None else None
    except Exception:
        return None

# for 2 features
def compat_l2(a, b):
    if a is None or b is None or len(a) != len(b):
        return 0.0
    # closer to each other, higher score
    return -float(np.linalg.norm(np.asarray(a, dtype=np.float32) - np.asarray(b, dtype=np.float32)))

def color_histogram_lab(rgb, bins=8):
    if rgb is None or rgb.size == 0:
        return None

    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB) # LAB
    hist_l = np.histogram(lab[:, :, 0], bins=bins, range=(0, 256))[0].astype(np.float32)
    hist_a = np.histogram(lab[:, :, 1], bins=bins, range=(0, 256))[0].astype(np.float32)
    hist_b = np.histogram(lab[:, :, 2], bins=bins, range=(0, 256))[0].astype(np.float32)
    h = np.concatenate([hist_l, hist_a, hist_b])
    # avoid 0/0
    return (h / (h.sum() + 1e-8))

def edge_features_4sides(rgb):
    if rgb is None or rgb.ndim < 2:
        return None

    if rgb.ndim == 2:
        rgb = np.stack([rgb, rgb, rgb], axis=-1)
    top = np.mean(rgb[0, :, :], axis=0)
    bottom = np.mean(rgb[-1, :, :], axis=0)
    left = np.mean(rgb[:, 0, :], axis=0)
    right = np.mean(rgb[:, -1, :], axis=0)
    return [top.astype(np.float32), bottom.astype(np.float32), left.astype(np.float32), right.astype(np.float32)]

# 8 angles
def rot_angle(img, angle_deg):
    if angle_deg == 0:
        return img
    h, w = img.shape[0], img.shape[1]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def rot90_k(img, k):
    return img if k == 0 else np.ascontiguousarray(np.rot90(img, k=k))

ANGLES = ANGLES_8 if USE_8_ANGLES else ANGLES_4
NUM_ROT = len(ANGLES)

# 4 angles
def _rotated_image(rgb, angle_deg):
    if angle_deg in (0, 90, 180, 270):
        k = ANGLES_4.index(angle_deg)
        return rot90_k(rgb, k)
    return rot_angle(rgb, angle_deg)

def get_combined_features(rgb_list, angles=None):
    if angles is None:
        angles = ANGLES
    n = len(rgb_list)
    combined_rot = []
    edge_4_rot = []
    for j in range(n):
        rgb = rgb_list[j]
        feats_rot = []
        edges_rot = []
        for ang in angles:
            rimg = _rotated_image(rgb, ang)
            hog = hog_descriptor(rimg)
            color = color_histogram_lab(rimg)
            edge_4 = edge_features_4sides(rimg)
            edge_global = np.concatenate(edge_4).flatten() if edge_4 else np.zeros(12, dtype=np.float32)
            if hog is None:
                hog = np.zeros(1, dtype=np.float32)
            if color is None:
                color = np.zeros(24, dtype=np.float32)
            c = np.concatenate([W_HOG * (hog.astype(np.float32)), W_COLOR * color, W_EDGE * edge_global])
            feats_rot.append(c)
            edges_rot.append(edge_4 if edge_4 else [np.zeros(3, dtype=np.float32)] * 4)
        combined_rot.append(feats_rot)
        edge_4_rot.append(edges_rot)
    return combined_rot, edge_4_rot

# for rotation 
def compat_combined(a, b):
    if a is None or b is None or len(a) != len(b):
        return 0.0
    return -float(np.linalg.norm(np.asarray(a, dtype=np.float32) - np.asarray(b, dtype=np.float32)))

def edge_compat_side(edge_a, edge_b):
    if edge_a is None or edge_b is None:
        return 0.0
    return -float(np.linalg.norm(np.asarray(edge_a, dtype=np.float32) - np.asarray(edge_b, dtype=np.float32)))


# fast, doing nothing
def geometry_grid_placement(n_pieces, W, H, hog_list=None):
    # print("geometry_grid_placement is used")
    if n_pieces <= 0:
        return []
    rows, cols = layout_rows_cols(n_pieces, W, H)
    tw, th = W / cols, H / rows
    order = np.arange(n_pieces)
    if hog_list is not None and len(hog_list) == n_pieces:
        norms = [np.linalg.norm(h) if h is not None else 0.0 for h in hog_list]
        order = np.argsort(norms)
    out = []
    for idx in range(n_pieces):
        k = order[idx]
        r, c = idx // cols, idx % cols
        out.append((k, c * tw + tw / 2.0, r * th + th / 2.0, 0.0))
    return out

# greedy
def geometry_placement_stage3(n_pieces, W, H, hog_list):
    if n_pieces <= 0 or hog_list is None or len(hog_list) != n_pieces:
        return geometry_grid_placement(n_pieces, W, H, hog_list)
    rows, cols = layout_rows_cols(n_pieces, W, H)
    tw, th = W / cols, H / rows
    n = n_pieces
    C = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i + 1, n):
            cij = compat_l2(hog_list[i], hog_list[j])
            C[i, j] = C[j, i] = cij
            
    seed = int(np.argmax([np.linalg.norm(h) if h is not None else 0.0 for h in hog_list]))
    rc, cc = rows // 2, cols // 2
    assign = {(rc, cc): seed}
    placed = {seed}
    def adj(r, c):
        out = []
        if r > 0: out.append((r - 1, c))
        if r < rows - 1: out.append((r + 1, c))
        if c > 0: out.append((r, c - 1))
        if c < cols - 1: out.append((r, c + 1))
        return out
    open_cells = set(adj(rc, cc))
    while len(placed) < n and open_cells:
        best_cell, best_piece, best_score = None, None, -1e9
        for (r, c) in sorted(open_cells):
            adjs = [(nr, nc) for nr, nc in adj(r, c) if (nr, nc) in assign]
            if not adjs:
                continue
            for p in range(n):
                if p in placed:
                    continue
                s = sum(C[p, assign[(nr, nc)]] for (nr, nc) in adjs)
                if s > best_score:
                    best_score, best_cell, best_piece = s, (r, c), p
        if best_cell is None:
            break
        r, c = best_cell
        assign[(r, c)] = best_piece
        placed.add(best_piece)
        open_cells.discard(best_cell)
        for nr, nc in adj(r, c):
            if (nr, nc) not in assign:
                open_cells.add((nr, nc))
    empty = [(r, c) for r in range(rows) for c in range(cols) if (r, c) not in assign]
    rest = [p for p in range(n) if p not in placed]
    for i, (r, c) in enumerate(empty):
        if i >= len(rest):
            break
        assign[(r, c)] = rest[i]
    return [(k, c * tw + tw / 2.0, r * th + th / 2.0, 0.0) for (r, c), k in assign.items()]

def _which_edge_toward(rc, cc, nr, nc):
    dr, dc = nr - rc, nc - cc
    if dr == -1 and dc == 0:   return (0, 1)  # our top vs their bottom
    if dr == 1 and dc == 0:   return (1, 0)
    if dr == 0 and dc == -1:  return (2, 3)  # our left vs their right
    if dr == 0 and dc == 1:   return (3, 2)
    return (0, 0)

def geometry_placement_stage3_with_rotation(n_pieces, W, H, rgb_list_uint8, try_rotation=True):
    if rgb_list_uint8 is None or len(rgb_list_uint8) != n_pieces:
        hog_list = [hog_descriptor(rgb_list_uint8[j]) if rgb_list_uint8 and j < len(rgb_list_uint8) else None for j in range(n_pieces)] if rgb_list_uint8 else None
        return geometry_placement_stage3(n_pieces, W, H, hog_list)
    if not try_rotation:
        combined, _ = get_combined_features(rgb_list_uint8, angles=[0])
        feat_list = [combined[j][0] for j in range(n_pieces)]
        return geometry_placement_stage3(n_pieces, W, H, feat_list)
    n = n_pieces
    combined_rot, edge_4_rot = get_combined_features(rgb_list_uint8, ANGLES)
    rows, cols = layout_rows_cols(n, W, H)
    tw, th = W / cols, H / rows
    areas = [contour_area(rgb_list_uint8[j]) for j in range(n)]
    norms0 = [np.linalg.norm(combined_rot[j][0]) for j in range(n)]
    seed = int(np.argmax(areas)) if max(areas) > 0 else int(np.argmax(norms0))
    rc, cc = rows // 2, cols // 2
    # print("geometry_placement_stage3_with_rotation is used")
    def adj(r, c):
        out = []
        if r > 0: out.append((r - 1, c))
        if r < rows - 1: out.append((r + 1, c))
        if c > 0: out.append((r, c - 1))
        if c < cols - 1: out.append((r, c + 1))
        return out

    def score_placement(assign, r, c, p, rot):
        s = 0.0
        for (nr, nc) in adj(r, c):
            if (nr, nc) not in assign:
                continue
            q, qrot = assign[(nr, nc)]
            s += compat_combined(combined_rot[p][rot], combined_rot[q][qrot])
            our_side, their_side = _which_edge_toward(r, c, nr, nc)
            s += EDGE_COMPAT_WEIGHT * edge_compat_side(edge_4_rot[p][rot][our_side], edge_4_rot[q][qrot][their_side])
        return s

    use_beam = (n <= BEAM_SEARCH_MAX_N and try_rotation)
    if use_beam:
        states = [({(rc, cc): (seed, 0)}, {seed}, set(adj(rc, cc)), 0.0)]
        while states and len(states[0][1]) < n:
            candidates = []
            for (assign, placed, open_cells, total_score) in states:
                for (r, c) in sorted(open_cells):
                    adjs = [(nr, nc) for nr, nc in adj(r, c) if (nr, nc) in assign]
                    if not adjs:
                        continue
                    for p in range(n):
                        if p in placed:
                            continue
                        for rot in range(NUM_ROT):
                            delta = score_placement(assign, r, c, p, rot)
                            new_assign = dict(assign)
                            new_assign[(r, c)] = (p, rot)
                            new_placed = set(placed) | {p}
                            new_open = set(open_cells) - {(r, c)} | {x for x in adj(r, c) if x not in new_assign}
                            candidates.append((new_assign, new_placed, new_open, total_score + delta))
            if not candidates:
                break
            candidates.sort(key=lambda x: -x[3])
            states = candidates[:BEAM_WIDTH]
        if states:
            assign, placed, _, _ = states[0]
        else:
            assign, placed = {(rc, cc): (seed, 0)}, {seed}
    else:
        assign = {(rc, cc): (seed, 0)}
        placed = {seed}
        open_cells = set(adj(rc, cc))
        while len(placed) < n and open_cells:
            best_cell, best_piece, best_rot, best_score = None, None, None, -1e9
            for (r, c) in sorted(open_cells):
                adjs = [(nr, nc) for nr, nc in adj(r, c) if (nr, nc) in assign]
                if not adjs:
                    continue
                for p in range(n):
                    if p in placed:
                        continue
                    for rot in range(NUM_ROT):
                        s = score_placement(assign, r, c, p, rot)
                        if s > best_score:
                            best_score, best_cell, best_piece, best_rot = s, (r, c), p, rot
            
            if best_cell is None:
                break
            r, c = best_cell
            assign[(r, c)] = (best_piece, best_rot)
            placed.add(best_piece)
            open_cells.discard(best_cell)
            for nr, nc in adj(r, c):
                if (nr, nc) not in assign:
                    open_cells.add((nr, nc))

    empty = [(r, c) for r in range(rows) for c in range(cols) if (r, c) not in assign]
    rest = [p for p in range(n) if p not in placed]
    for i, (r, c) in enumerate(empty):
        if i >= len(rest):
            break
        assign[(r, c)] = (rest[i], 0)
    return [(k, c * tw + tw / 2.0, r * th + th / 2.0, float(ANGLES[rot])) for (r, c), (k, rot) in assign.items()]

# Backward-compat: predictor uses this for grid path
def extract_hog_per_fragment(rgb):
    return hog_descriptor(rgb)

def parse_canvas_from_parameters(parameters_txt):
    kv = {}
    for line in parameters_txt.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        parts = s.split(maxsplit=1)
        if len(parts) >= 2:
            kv[parts[0]] = parts[1]

    for w_key, h_key in (("solution_width", "solution_height"), ("canvas_width", "canvas_height")):
        if w_key in kv and h_key in kv:
            try:
                W, H = float(kv[w_key]), float(kv[h_key])
                if W > 0 and H > 0:
                    return W, H
            except ValueError:
                pass

    for size_key in ("reference_size", "square_size"):
        if size_key in kv:
            try:
                side = float(kv[size_key])
                if side > 0:
                    return side, side
            except ValueError:
                pass
    return None, None


def rgb_tensor_to_uint8(t):
    if hasattr(t, "is_cuda") and t.is_cuda:
        t = t.cpu()
    arr = t.numpy() if hasattr(t, "numpy") else np.array(t)
    if arr.ndim == 3 and arr.shape[0] == 3:
        arr = np.transpose(arr, (1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406], dtype=arr.dtype)
    std = np.array([0.229, 0.224, 0.225], dtype=arr.dtype)
    return (np.clip(arr * std + mean, 0, 1) * 255).astype(np.uint8)


def _predict_single_puzzle(batch, idx, ds_name):
    puzzle_key = batch["puzzle_key"][idx]
    parameters_txt = batch["parameters_txt"][idx]

    W, H = parse_canvas_from_parameters(parameters_txt)
    if W is None or H is None:
        return None

    fids = batch["fragment_ids"][idx].tolist()
    n = len(fids)
    rgb_tensor_list = batch["rgb"][idx]
    rgb_list_uint8 = [rgb_tensor_to_uint8(rgb_tensor_list[j]) for j in range(n)]

    if n > STAGE3_FAST_PATH_MAX_N:
        combined_0, _ = get_combined_features(rgb_list_uint8, angles=[0])
        feat_list = [combined_0[j][0] for j in range(n)]
        poses = geometry_grid_placement(n, W, H, feat_list)
    else:
        poses = geometry_placement_stage3_with_rotation(
            n, W, H, rgb_list_uint8, try_rotation=(n <= STAGE3_FAST_PATH_MAX_N)
        )

    pred_poses = {fids[k]: (x, y, theta) for (k, x, y, theta) in poses}

    gt_xy = batch["xy"][idx].cpu()
    gt_rot = batch["rotation_deg"][idx].cpu()
    gt_poses = {}
    output_rows = []
    for j, fid in enumerate(fids):
        px, py, theta = pred_poses[fid]
        gx = float(gt_xy[j, 0].item()) * W
        gy = float(gt_xy[j, 1].item()) * H
        gtheta = float(gt_rot[j].item())
        gt_poses[fid] = {"x": gx, "y": gy, "theta": gtheta}
        output_rows.append((ds_name, puzzle_key, fid, px, py, theta))

    key = (ds_name, puzzle_key)
    cache_item = (key, dict(pred_poses), rgb_list_uint8, W, H, list(fids))
    return key, gt_poses, pred_poses, output_rows, cache_item


def _save_pose_files(pose_output_lines):
    by_puzzle = defaultdict(list)
    for ds_name, puzzle_key, fid, px, py, theta in pose_output_lines:
        by_puzzle[(ds_name, puzzle_key)].append((fid, px, py, theta))

    for (ds_name, puzzle_key), rows in by_puzzle.items():
        safe_key = f"{ds_name}_{puzzle_key.replace('/', '_')}"
        out_path = OUTPUT_DIR / f"{safe_key}_poses.txt"
        with open(out_path, "w") as f:
            for fid, px, py, theta in sorted(rows, key=lambda item: item[0]):
                f.write(f"{fid} {px:.4f} {py:.4f} {theta:.4f}\n")


run_times = []
all_gt_poses = {}
all_pred_poses = {}
pose_output_lines = []
viz_data = None
viz_data_by_ds = {}
# Per-group timing: key = (ds_name, puzzle_key), value = seconds
group_times = {}

for run_id in range(1):
    t0 = time.perf_counter()
    val_loaders = make_val_loaders()

    all_gt_poses.clear()
    all_pred_poses.clear()
    pose_output_lines.clear()
    viz_data = None
    viz_data_by_ds = {}
    group_times = {}

    for ds_name, loader in val_loaders.items():
        print(f"[{ds_name}] processing...", flush=True)
        processed = 0

        for batch in loader:
            batch_size = len(batch["puzzle_key"])
            for i in range(batch_size):
                t_group_0 = time.perf_counter()
                packed = _predict_single_puzzle(batch, i, ds_name)
                if packed is None:
                    continue

                key, gt_poses, pred_poses, rows, cache_item = packed
                group_times[key] = time.perf_counter() - t_group_0
                all_gt_poses[key] = gt_poses
                all_pred_poses[key] = pred_poses
                pose_output_lines.extend(rows)

                if processed == 0:
                    viz_data = cache_item
                if ds_name not in viz_data_by_ds:
                    viz_data_by_ds[ds_name] = cache_item

                processed += 1

    elapsed = time.perf_counter() - t0
    run_times.append(elapsed)

# if len(run_times) >= 3:
for i, t in enumerate(run_times, 1):
    print(f"Run {i}: {t:.2f} s", flush=True)
print(f"Average: {sum(run_times)/len(run_times):.2f} s", flush=True)
print(f"device: {DEVICE}")

_save_pose_files(pose_output_lines)

## 8. Qualitative Results of Solution

In [ ]:
# Todo: Add your code here

## 8. Quantitative Results 
Implement translation and rotation errors.


In [ ]:
def rmse_translation(gt_poses, pred_poses):
    """Compute RMSE of translation (pixels) over pieces present in gt_poses."""
    sq_errors = []
    for pid, gt in gt_poses.items():
        if pid not in pred_poses:
            continue
        px, py, _ = pred_poses[pid]
        dx = px - gt["x"]
        dy = py - gt["y"]
        sq_errors.append(dx * dx + dy * dy)
    if not sq_errors:
        return float("nan")
    mse = sum(sq_errors) / len(sq_errors)
    return math.sqrt(mse)


def rmse_rotation(gt_poses, pred_poses):
    """Compute RMSE of rotation (degrees) over pieces."""
    sq_errors = []
    for pid, gt in gt_poses.items():
        if pid not in pred_poses:
            continue
        _, _, ptheta = pred_poses[pid]
        dtheta = ptheta - gt["theta"]
        dtheta = (dtheta + 180.0) % 360.0 - 180.0
        sq_errors.append(dtheta * dtheta)
    if not sq_errors:
        return float("nan")
    mse = sum(sq_errors) / len(sq_errors)
    return math.sqrt(mse)


def neighbour_f1(gt_poses, pred_poses, threshold_px=50.0):
    valid_ids = [pid for pid in gt_poses if pid in pred_poses]
    if len(valid_ids) < 2:
        return float("nan")

    thr2 = float(threshold_px) ** 2
    f1_scores = []
    for anchor in valid_ids:
        gx, gy = gt_poses[anchor]["x"], gt_poses[anchor]["y"]
        px, py = pred_poses[anchor][0], pred_poses[anchor][1]

        gt_neighbors = {
            pid for pid in valid_ids
            if pid != anchor and (gt_poses[pid]["x"] - gx) ** 2 + (gt_poses[pid]["y"] - gy) ** 2 < thr2
        }
        pred_neighbors = {
            pid for pid in valid_ids
            if pid != anchor and (pred_poses[pid][0] - px) ** 2 + (pred_poses[pid][1] - py) ** 2 < thr2
        }

        if not gt_neighbors and not pred_neighbors:
            f1_scores.append(1.0)
            continue
        if not gt_neighbors or not pred_neighbors:
            f1_scores.append(0.0)
            continue

        overlap = len(gt_neighbors & pred_neighbors)
        precision = overlap / len(pred_neighbors)
        recall = overlap / len(gt_neighbors)
        f1_scores.append((2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0)

    return float(np.mean(f1_scores))


rmse_t_by_puzzle = []
rmse_r_by_puzzle = []
nf1_by_puzzle = []
per_dataset = defaultdict(lambda: {"t": [], "r": [], "f1": []})

for key, gt_pose_dict in all_gt_poses.items():
    pred_pose_dict = all_pred_poses.get(key, {})

    t_rmse = rmse_translation(gt_pose_dict, pred_pose_dict)
    r_rmse = rmse_rotation(gt_pose_dict, pred_pose_dict)
    f1_val = neighbour_f1(gt_pose_dict, pred_pose_dict)

    if not math.isnan(t_rmse):
        rmse_t_by_puzzle.append(t_rmse)
        per_dataset[key[0]]["t"].append(t_rmse)
    if not math.isnan(r_rmse):
        rmse_r_by_puzzle.append(r_rmse)
        per_dataset[key[0]]["r"].append(r_rmse)
    if not math.isnan(f1_val):
        nf1_by_puzzle.append(f1_val)
        per_dataset[key[0]]["f1"].append(f1_val)

print(f"RMSE-T (translation, pixels): {np.mean(rmse_t_by_puzzle) if rmse_t_by_puzzle else float('nan'):.4f}")
print(f"RMSE-R (rotation, degrees):   {np.mean(rmse_r_by_puzzle) if rmse_r_by_puzzle else float('nan'):.4f}")
print(f"Neighbour-F1: {np.mean(nf1_by_puzzle) if nf1_by_puzzle else float('nan'):.4f}")
print(f"Over {len(rmse_t_by_puzzle)} puzzles.")

print("Per-difficulty:")
for ds_name in sorted(per_dataset.keys()):
    vals = per_dataset[ds_name]
    n = len(vals["t"])
    t_mean = np.mean(vals["t"]) if vals["t"] else float("nan")
    r_mean = np.mean(vals["r"]) if vals["r"] else float("nan")
    f1_mean = np.mean(vals["f1"]) if vals["f1"] else float("nan")
    print(f"  {ds_name}: RMSE-T = {t_mean:.4f}, RMSE-R = {r_mean:.4f}, NF1 = {f1_mean:.4f}, n = {n}")

if "run_times" in dir() and len(run_times) >= 3:
    print("Timing:")
    for run_idx, sec in enumerate(run_times, 1):
        print(f"  run_{run_idx}: {sec:.2f} s")
    print(f"  mean: {np.mean(run_times):.2f} s")
print(f"device: {DEVICE}")

## 8. Time Script (Do not remove)

In [ ]:
script_end = time.perf_counter()
total_time = script_end - script_start

print(f"Total execution time: {total_time:.6f} seconds")